# Plug in a driver, step by step

A walk-through for someone who would like to connect a microscope to the
ZMART Controller. It connects to the mock driver first, looks at what a
driver is made of, writes a small driver of your own for a pretend
microscope, checks it, and installs it so this computer knows it by name.
The [README](README.md) of this part is the complete reference: every
method a driver must have, what each must hand back, and the rules behind
them.

**Before you start**

- Python 3.11 or newer, with the controller installed:
  `pip install "git+https://github.com/thomdehoog/ZMART-controller"`
- No microscope. Everything here runs on a laptop.

Run the cells one at a time, from top to bottom.

## The idea

A workflow written for one microscope does not run on another, because every
microscope software speaks its own language. The controller stands in
between. Your workflow only ever speaks the controller's short list of
commands, and a **driver** translates each command for one particular
microscope.

```
your workflow ──► zmart_controller ──► driver ──► vendor software ──► microscope
```

A driver is two files. `zmart_driver.json` holds the driver's name and how
to reach the microscope. `zmart_driver.py` holds a class, `ZmartDriver`,
with **one method per command**: `get_xyz`, `set_xyz`, `acquire`, and so
on. The controller calls those methods. Plugging in a driver means telling
the controller, once, where the two files are on this computer. That is the
last step here. First we see what the methods are.

## Step 1: connect to the mock driver

The controller comes with one driver: the mock, a simulated microscope. It
is already plugged in, so connect to it and look around.

In [ ]:
import zmart_controller

zmart_controller.get_instruments()

In [ ]:
zmart_controller.set_instrument("mock", {"output_root": "images"})
zmart_controller.get_actuators()["content"]

These are the motors that can move each axis. z has two: a motor for long
moves and a piezo for fine, fast steps.

Now move the stage and take a picture. Every command answers in the same
shape, `{"success": ..., "content": ...}`.

In [ ]:
zmart_controller.set_xyz(100, 50, 0)
answer = zmart_controller.acquire(position_label="A1")
answer["content"]["files"]

In [ ]:
zmart_controller.disconnect()

Part 2, [Use the controller](../2_use_the_controller/tutorial.ipynb), goes
through all the commands. Here we look at the other side: the driver.

## Step 2: look at what a driver is

The controller ships a driver to copy, in the folder
`zmart_controller/template`. It is the two files, with every method still
to be written. Look at the settings file first.

In [ ]:
from pathlib import Path

import zmart_controller.template

template = Path(zmart_controller.template.__file__).parent
print((template / "zmart_driver.json").read_text())

Every value says what to put there. The class file next to it,
`zmart_driver.py`, is the `ZmartDriver` with one method per command. Open
it: each method says in its docstring what it must hand back, and raises
`NotImplementedError` until you write it. These method names are the whole
connection between the controller and a microscope.

## Step 3: write a driver of your own

The cells below make a folder called `pretend_scope` next to this notebook
and write the two files into it (that is what the first line, `%%writefile`,
does). It is a complete driver for a microscope that only exists in memory.
It remembers where its stage is and what its exposure is. Each "picture" is
a small text file that says where and how it was taken.

The settings file first. The pretend microscope needs nothing to be
reached, so the connection only says which instrument it is.

In [ ]:
from pathlib import Path

Path("pretend_scope").mkdir(exist_ok=True)

In [ ]:
%%writefile pretend_scope/zmart_driver.json
{
  "name": "pretend-scope",
  "driver": "zmart_driver.py",
  "connection": {
    "microscope": "pretend-01"
  }
}

Now the class. Every method hands back `True` and the values, or `False`
and a message; the controller turns that into `{"success": ..., "content": ...}`
for the workflow.

In [ ]:
%%writefile pretend_scope/zmart_driver.py
"""A ZMART driver for a pretend microscope that only exists in memory."""

import json
from pathlib import Path

# How far each axis can travel, in micrometres from the origin.
TRAVEL = {"x": [-1000.0, 1000.0], "y": [-1000.0, 1000.0], "z": [-100.0, 100.0]}

# Half a field of view in x and y. A picture taken at the edge of the travel
# still shows this much further out. This microscope takes no z-stacks, so 0 in z.
HALF_FIELD = {"x": 50.0, "y": 50.0, "z": 0.0}


class ZmartDriver:
    """One connected pretend microscope. It keeps its stage and its exposure on self."""

    def __init__(self, connection):
        self.position = {"x": 0.0, "y": 0.0, "z": 0.0}
        self.settings = {"exposure_ms": 10.0}
        self.output_root = Path("pretend_scope_images")
        self.output_root.mkdir(exist_ok=True)

    def disconnect(self):
        pass  # nothing to close in memory

    def get_info(self):
        description = (
            "A pretend microscope that lives in memory. One motor per axis. "
            "exposure_ms is the exposure time in milliseconds. +z points towards the objective."
        )
        return True, (str(self.output_root), description)

    def get_actuators(self):
        return True, (["motor"], ["motor"], ["motor"])

    def get_xyz(self, with_actuators):
        p = self.position
        canvas = []
        for axis, (low, high) in TRAVEL.items():
            canvas += [low - HALF_FIELD[axis], high + HALF_FIELD[axis]]
        return True, (p["x"], p["y"], p["z"], "motor", "motor", "motor", tuple(canvas))

    def set_xyz(self, x, y, z, with_actuators):
        for axis, value in {"x": x, "y": y, "z": z}.items():
            low, high = TRAVEL[axis]
            if not low <= value <= high:
                raise ValueError(f"{axis} = {value} um is outside the travel, {low} to {high} um")
        self.position = {"x": x, "y": y, "z": z}
        return True, ("motor", "motor", "motor")

    def get_state(self):
        return True, (dict(self.settings), {"objective": "10x"})

    def set_state(self, changeable):
        applied = {}
        for name, value in changeable.items():
            if name not in self.settings:
                raise ValueError(f"unknown setting {name!r}")
            if value <= 0:
                return False, f"{name} stayed {self.settings[name]}: the camera ignores {value}"
            self.settings[name] = value
            applied[name] = value
        return True, applied

    def get_acquisition_settings(self):
        return True, {"format": {"options": ["json"], "active": "json"}}

    def acquire(self, position_label, acquisition_settings):
        for name, value in (acquisition_settings or {}).items():
            if name != "format" or value != "json":
                raise ValueError(f"unknown acquisition setting {name!r} = {value!r}")
        path = self.output_root / f"{position_label}.json"
        path.write_text(json.dumps({"position": self.position, "settings": self.settings}))
        p = self.position
        plane = {"path": str(path), "c": 0, "z": 0, "t": 0, "x_um": p["x"], "y_um": p["y"], "z_um": p["z"]}
        return True, ([str(path)], [plane])

    def get_procedures(self):
        return True, {}

    def run_procedure(self, procedure):
        raise ValueError(f"unknown procedure {procedure.get('name')!r}; this microscope has none")


A few things to notice, because a real driver does the same:

- **Making the class opens the connection.** Whatever you keep on `self` is
  there for every later command. A real driver keeps its connection to the
  vendor software there.
- **`set_xyz` refuses a position outside the travel by raising
  `ValueError`.** It never moves "as far as it can". A workflow that asked for
  somewhere else must hear about it.
- **`acquire` says where it saved the picture** (`files`) and where on the
  sample it was taken (`planes`). That is how a workflow finds its pictures on
  any microscope.
- **The canvas is the travel widened by half a field**, because a picture
  taken at the edge of the travel still shows a little beyond it.

## Step 4: connect to it

While you write a driver, there is no need to plug it in yet. Load it from
its folder and hand it to `set_instrument`, and connect exactly as you did
with the mock.

In [ ]:
pretend = zmart_controller.load_driver("pretend_scope/zmart_driver.json")
zmart_controller.set_instrument(pretend)

zmart_controller.set_xyz(100, 50, 0)
answer = zmart_controller.acquire(position_label="A1")
answer["content"]["files"]

The "picture" records the position and the exposure.

In [ ]:
print(open(answer["content"]["files"][0]).read())

Now ask for a position outside the travel. Your method raises, and the
answer is `success: False` with the error text as the content. Then ask for
an exposure of zero: the method hands back `False` and its own message.

In [ ]:
zmart_controller.set_xyz(5000, 0, 0)

In [ ]:
zmart_controller.set_state({"changeable": {"exposure_ms": 0}})

Three things can come out of a method. It hands back `True` and the values:
`success: True`. It hands back `False` and a message, because the
microscope did not do it: `success: False` with that message. Or it raises,
for a request that is wrong or a bug: `success: False` with the error text.

## Step 5: check that it fits

The controller can check a driver's answers against the requirements.
`validate_driver` connects, calls every `get_*` method, and lists any
problem in plain words. An empty list means every answer has the right shape.

In [ ]:
zmart_controller.validate_driver("pretend_scope/zmart_driver.json")

It moves nothing and acquires nothing, so check one acquisition separately.

In [ ]:
answer = zmart_controller.acquire(position_label="A2")
zmart_controller.check_acquire_answer(answer)

If a method hands back the wrong thing, `validate_driver` says so in one
sentence per problem, naming the command and what is missing. This is the
loop to work in when you write a driver: write a method, validate, read the
problems, repeat.

This is the loop to work in when you write a driver: write a method,
validate, read the problems, repeat.

## Step 6: plug it in

So far you loaded the driver from its folder yourself. To use the driver
like any other, plug it in: install it once on this computer by pointing
the controller at its settings file.

In [ ]:
zmart_controller.register_driver("pretend_scope/zmart_driver.json")
zmart_controller.get_instruments()

From now on, every Python session on this computer can connect to it by name.

In [ ]:
zmart_controller.set_instrument("pretend-scope")
zmart_controller.get_info()["content"]["description"]

In [ ]:
zmart_controller.disconnect()

This tutorial's driver does not need to stay on the list. Take it off again.

In [ ]:
zmart_controller.remove_driver("pretend-scope")
zmart_controller.get_instruments()

## Step 7: from a pretend microscope to a real one

Your two files already work with every workflow written for the controller.
For a real microscope, each method does the same job, but by talking to the
vendor software instead of to `self`:

- **Making the class** starts or logs into the vendor software with what is
  in the connection, and loads what was measured once on this microscope:
  the origin, the travel limits, the calibration. These live on the
  microscope computer, in the folder that
  `zmart_controller.registry.config_root()` names.
- **`set_xyz`** checks the limits, sends the move, and reads the position
  back until it matches, because microscope software often says "done" before
  the stage has stopped.
- **`acquire`** starts the capture, waits until the vendor's file is
  complete, turns it into OME-TIFF or OME-Zarr lined up with the stage, and
  returns `files` and `planes`.

Once the class grows beyond one file, split the vendor-talking code into
modules of its own next to it; `zmart_driver.json` keeps pointing at the
file with the class.
[The anatomy of a ZMART driver](https://github.com/thomdehoog/ZMART-drivers/blob/main/docs/driver-anatomy.md)
describes how our own drivers are built inside, and the mock driver is a
complete example of it.

## When something goes wrong

| Message | Meaning | What to do |
|---|---|---|
| `ValueError: ... zmart_driver.py is missing` | The JSON's `driver` does not point at a file | Check the path in `zmart_driver.json`; it is relative to the JSON's folder |
| `ValueError: ... must define the class ZmartDriver` | The class file has no class of that name | Name the class exactly `ZmartDriver` |
| `NotImplementedError: ZmartDriver.<method>: ...` | That method is still the template's | Write it |
| `validate_driver` lists problems | Each sentence names the command and what is missing | Fix that method and validate again |
| `success: False` with your own error text | A method raised | The content points at your code |

## Good to know

- **The method names are the contract.** The controller calls them by name.
  Keep them exactly as in the template, and give your helpers other names.
- **`True` and the values, or `False` and a message.** Raise `ValueError`
  for a request that is wrong. Either way the workflow gets `success: False`
  with a text that says what happened.
- **Positions are micrometres from the origin.** In a saved picture, right is
  +x and down is +y, on every microscope. A real driver turns the vendor's own
  numbers and image orientation into this frame.
- **Never overwrite a picture.** The pretend driver above does, to stay
  short. A real one, like the mock, saves a second acquisition with the same
  label as `A1_001` instead.

## Where to go next

- The [README](README.md): the full requirements for every method.
- Part 2, [Use the controller](../2_use_the_controller/tutorial.ipynb):
  every command, on the mock.

---

MIT license. Thom de Hoog, Center for Microscopy and Image Analysis (ZMB), University of Zurich. thom.dehoog@zmb.uzh.ch, thomdehoog@gmail.com.